In [ ]:
# STORAGE FOR COMPARISON TABLE
results = []

## Linear Regression

In [ ]:
import random
import warnings
import torch
import pandas as pd
import numpy as np
import pickle

from sklearn.model_selection import KFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import r2_score, make_scorer, mean_squared_error


# Machine Learning Models
from sklearn.linear_model import LinearRegression


# REPRODUCIBILITY
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

warnings.filterwarnings("ignore")

# LOAD DATA
df = pd.read_csv(
    "../Unified_Resume_Salary_Dataset_ori.csv"
)

with open("../master_vocab.pkl", "rb") as f:
    master_vocab = pickle.load(f)

train_idx = np.load("../train_idx_ori.npy")
test_idx  = np.load("../test_idx_ori.npy")

X_text = df["Aligned_Text"]
y = df["Log_Salary"]

X_train = X_text.iloc[train_idx]
y_train = y.iloc[train_idx].values

# PIPELINE
tfidf_config = dict(
    vocabulary=master_vocab,
    ngram_range=(1, 3),
    token_pattern=r'(?u)\.?[a-z0-9][a-z0-9\+\#\.]*',
    norm=None,
    binary=True,
    min_df=5
)

model = Pipeline([
    ("tfidf", TfidfVectorizer(**tfidf_config)),
    ("model", LinearRegression())
])

# CROSS VALIDATION SETUP
cv = KFold(n_splits=5, shuffle=True, random_state=seed)

scoring = {
    "r2": "r2",
    "rmse": "neg_root_mean_squared_error"
}


# RUN CROSS VALIDATION
cv_results = cross_validate(
    model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)

# METRICS (LOG SCALE - CORRECT)
train_r2 = np.mean(cv_results["train_r2"])
val_r2   = np.mean(cv_results["test_r2"])

train_rmse = -np.mean(cv_results["train_rmse"])
val_rmse   = -np.mean(cv_results["test_rmse"])

gap = train_r2 - val_r2

# PRINT PER MODEL SUMMARY
print("\n" + "="*80)
print(" LINEAR REGRESSION BASELINE - CROSS VALIDATION RESULTS (LOG SCALE)")
print("="*80)
print(f"Train R²   : {train_r2:.4f}")
print(f"Test  R²   : {val_r2:.4f}")
print(f"Gap        : {gap:.4f}")
print(f"Train RMSE : {train_rmse:.4f}")
print(f"Test  RMSE : {val_rmse:.4f}")
print("="*80)

# STORE RESULTS FOR FINAL TABLE
results.append({
    "Model": "Linear Regression",
    "Train_R2": train_r2,
    "Val_R2": val_r2,
    "Gap_R2": gap,
    "Train_RMSE": train_rmse,
    "Val_RMSE": val_rmse
})


 LINEAR REGRESSION BASELINE - CROSS VALIDATION RESULTS (LOG SCALE)
Train R²   : 0.3461
Test  R²   : 0.3198
Gap        : 0.0263
Train RMSE : 0.2713
Test  RMSE : 0.2766


## LightGBM

In [ ]:
import random
import warnings
import torch
import pandas as pd
import numpy as np
import pickle

from sklearn.model_selection import KFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import r2_score, make_scorer, mean_squared_error

# Machine Learning Models
import lightgbm as lgb

# REPRODUCIBILITY
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

warnings.filterwarnings("ignore")

# LOAD DATA
df = pd.read_csv(
    "../Unified_Resume_Salary_Dataset_ori.csv"
)

with open("../master_vocab.pkl", "rb") as f:
    master_vocab = pickle.load(f)

train_idx = np.load("../train_idx_ori.npy")
test_idx  = np.load("../test_idx_ori.npy")

X_text = df["Aligned_Text"]
y = df["Log_Salary"]

X_train = X_text.iloc[train_idx]
y_train = y.iloc[train_idx].values

# PIPELINE
tfidf_config = dict(
    vocabulary=master_vocab,
    ngram_range=(1, 3),
    token_pattern=r'(?u)\.?[a-z0-9][a-z0-9\+\#\.]*',
    norm=None,
    binary=True,
    min_df=5
)

model = Pipeline([
    ("tfidf", TfidfVectorizer(**tfidf_config)),
    ("model", lgb.LGBMRegressor(
        random_state=seed,
        n_jobs=-1,
        verbose=-1
    ))
])

# CROSS VALIDATION SETUP
cv = KFold(n_splits=5, shuffle=True, random_state=seed)

scoring = {
    "r2": "r2",
    "rmse": "neg_root_mean_squared_error"
}

# RUN CROSS VALIDATION
cv_results = cross_validate(
    model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)

# METRICS (LOG SCALE - CORRECT)
train_r2 = np.mean(cv_results["train_r2"])
val_r2   = np.mean(cv_results["test_r2"])

train_rmse = -np.mean(cv_results["train_rmse"])
val_rmse   = -np.mean(cv_results["test_rmse"])

gap = train_r2 - val_r2

# PRINT PER MODEL SUMMARY
print("\n" + "="*80)
print(" LIGHTGBM BASELINE - CROSS VALIDATION RESULTS (LOG SCALE)")
print("="*80)
print(f"Train R²   : {train_r2:.4f}")
print(f"Test  R²   : {val_r2:.4f}")
print(f"Gap        : {gap:.4f}")
print(f"Train RMSE : {train_rmse:.4f}")
print(f"Test  RMSE : {val_rmse:.4f}")
print("="*80)

# STORE RESULTS FOR FINAL TABLE
results.append({
    "Model": "LightGBM",
    "Train_R2": train_r2,
    "Val_R2": val_r2,
    "Gap_R2": gap,
    "Train_RMSE": train_rmse,
    "Val_RMSE": val_rmse
})

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/v


 LIGHTGBM BASELINE - CROSS VALIDATION RESULTS (LOG SCALE)
Train R²   : 0.5644
Test  R²   : 0.4814
Gap        : 0.0830
Train RMSE : 0.2214
Test  RMSE : 0.2415


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


## Catboost

In [ ]:
import random
import warnings
import torch
import pandas as pd
import numpy as np
import pickle

from sklearn.model_selection import KFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import r2_score, make_scorer, mean_squared_error

# Machine Learning Models
from catboost import CatBoostRegressor

# REPRODUCIBILITY
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

warnings.filterwarnings("ignore")

# LOAD DATA
df = pd.read_csv(
    "../Unified_Resume_Salary_Dataset_ori.csv"
)

with open("../master_vocab.pkl", "rb") as f:
    master_vocab = pickle.load(f)

train_idx = np.load("../train_idx_ori.npy")
test_idx  = np.load("../test_idx_ori.npy")

X_text = df["Aligned_Text"]
y = df["Log_Salary"]

X_train = X_text.iloc[train_idx]
y_train = y.iloc[train_idx].values


# PIPELINE
tfidf_config = dict(
    vocabulary=master_vocab,
    ngram_range=(1, 3),
    token_pattern=r'(?u)\.?[a-z0-9][a-z0-9\+\#\.]*',
    norm=None,
    binary=True,
    min_df=5
)

model = Pipeline([
    ("tfidf", TfidfVectorizer(**tfidf_config)),
    ("model", CatBoostRegressor(random_state=42, verbose=False))
])

# CROSS VALIDATION SETUP
cv = KFold(n_splits=5, shuffle=True, random_state=seed)

scoring = {
    "r2": "r2",
    "rmse": "neg_root_mean_squared_error"
}

# RUN CROSS VALIDATION
cv_results = cross_validate(
    model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)

# METRICS (LOG SCALE - CORRECT)
train_r2 = np.mean(cv_results["train_r2"])
val_r2   = np.mean(cv_results["test_r2"])

train_rmse = -np.mean(cv_results["train_rmse"])
val_rmse   = -np.mean(cv_results["test_rmse"])

gap = train_r2 - val_r2

# PRINT PER MODEL SUMMARY
print("\n" + "="*80)
print(" CATBOOST BASELINE - CROSS VALIDATION RESULTS (LOG SCALE)")
print("="*80)
print(f"Train R²   : {train_r2:.4f}")
print(f"Test  R²   : {val_r2:.4f}")
print(f"Gap        : {gap:.4f}")
print(f"Train RMSE : {train_rmse:.4f}")
print(f"Test  RMSE : {val_rmse:.4f}")
print("="*80)

# STORE RESULTS FOR FINAL TABLE
results.append({
    "Model": "CatBoost",
    "Train_R2": train_r2,
    "Val_R2": val_r2,
    "Gap_R2": gap,
    "Train_RMSE": train_rmse,
    "Val_RMSE": val_rmse
})


 CATBOOST BASELINE - CROSS VALIDATION RESULTS (LOG SCALE)
Train R²   : 0.6229
Test  R²   : 0.4922
Gap        : 0.1308
Train RMSE : 0.2060
Test  RMSE : 0.2390


## SVR

In [ ]:
import random
import warnings
import torch
import pandas as pd
import numpy as np
import pickle

from sklearn.model_selection import KFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import r2_score, make_scorer, mean_squared_error

# Machine Learning Models
from sklearn.svm import SVR

# REPRODUCIBILITY
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

warnings.filterwarnings("ignore")

# LOAD DATA
df = pd.read_csv(
    "../Unified_Resume_Salary_Dataset_ori.csv"
)

with open("../master_vocab.pkl", "rb") as f:
    master_vocab = pickle.load(f)

train_idx = np.load("../train_idx_ori.npy")
test_idx  = np.load("../test_idx_ori.npy")

X_text = df["Aligned_Text"]
y = df["Log_Salary"]

X_train = X_text.iloc[train_idx]
y_train = y.iloc[train_idx].values

# PIPELINE
tfidf_config = dict(
    vocabulary=master_vocab,
    ngram_range=(1, 3),
    token_pattern=r'(?u)\.?[a-z0-9][a-z0-9\+\#\.]*',
    norm=None,
    binary=True,
    min_df=5
)

model = Pipeline([
    ("tfidf", TfidfVectorizer(**tfidf_config)),
    ("model", SVR())
])

# CROSS VALIDATION SETUP
cv = KFold(n_splits=5, shuffle=True, random_state=seed)

scoring = {
    "r2": "r2",
    "rmse": "neg_root_mean_squared_error"
}

# RUN CROSS VALIDATION
cv_results = cross_validate(
    model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)

# METRICS (LOG SCALE - CORRECT)
train_r2 = np.mean(cv_results["train_r2"])
val_r2   = np.mean(cv_results["test_r2"])

train_rmse = -np.mean(cv_results["train_rmse"])
val_rmse   = -np.mean(cv_results["test_rmse"])

gap = train_r2 - val_r2

# PRINT PER MODEL SUMMARY
print("\n" + "="*80)
print(" SVR BASELINE - CROSS VALIDATION RESULTS (LOG SCALE)")
print("="*80)
print(f"Train R²   : {train_r2:.4f}")
print(f"Test  R²   : {val_r2:.4f}")
print(f"Gap        : {gap:.4f}")
print(f"Train RMSE : {train_rmse:.4f}")
print(f"Test  RMSE : {val_rmse:.4f}")
print("="*80)

# STORE RESULTS FOR FINAL TABLE
results.append({
    "Model": "SVR",
    "Train_R2": train_r2,
    "Val_R2": val_r2,
    "Gap_R2": gap,
    "Train_RMSE": train_rmse,
    "Val_RMSE": val_rmse
})


 SVR BASELINE - CROSS VALIDATION RESULTS (LOG SCALE)
Train R²   : 0.7540
Test  R²   : 0.4746
Gap        : 0.2794
Train RMSE : 0.1664
Test  RMSE : 0.2431


## Set Transformer

In [ ]:
import random
import warnings
import copy
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import pickle

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.base import BaseEstimator, RegressorMixin
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore")

# SEED (REPRODUCIBILITY)
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# DATA
df = pd.read_csv("../Unified_Resume_Salary_Dataset_ori.csv")

with open("../master_vocab.pkl", "rb") as f:
    master_vocab = pickle.load(f)

train_idx = np.load("../train_idx_ori.npy")
test_idx = np.load("../test_idx_ori.npy")

X_text = df["Aligned_Text"]
y = df["Log_Salary"].values

X_train_text = X_text.iloc[train_idx]
y_train = y[train_idx]
X_test_text = X_text.iloc[test_idx]
y_test = y[test_idx]

# COUNT VECTOR
vectorizer = CountVectorizer(
    vocabulary=master_vocab,
    ngram_range=(1, 3),
    token_pattern=r'(?u)\.?[a-z0-9][a-z0-9\+\#\.]*'
)

X_train_count = vectorizer.fit_transform(X_train_text)

# LOAD ONTOLOGY EMBEDDINGS
emb_path = "../ontology_embeddings.npy"
ontology_embs = torch.tensor(np.load(emb_path)).to(device)

def create_set_embeddings(X_count_matrix, embs_tensor, max_skills=30):
    X_count_matrix = X_count_matrix.tocsr()
    embs = embs_tensor.cpu().numpy()

    n, d = X_count_matrix.shape[0], embs.shape[1]
    X_set = np.zeros((n, max_skills, d), dtype=np.float32)

    for i in range(n):
        start = X_count_matrix.indptr[i]
        end = X_count_matrix.indptr[i + 1]
        idx = X_count_matrix.indices[start:end][:max_skills]

        if len(idx) > 0:
            X_set[i, :len(idx)] = embs[idx]

    return X_set

# MUST exist from previous step
X_train_set = create_set_embeddings(X_train_count, ontology_embs, max_skills=30)

# MODEL
class ISAB(nn.Module):
    def __init__(self, dim, heads, num_inducing=32):
        super().__init__()
        self.inducing = nn.Parameter(torch.randn(1, num_inducing, dim))
        self.mha1 = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.mha2 = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.ln1 = nn.LayerNorm(dim)
        self.ln2 = nn.LayerNorm(dim)

    def forward(self, x):
        H, _ = self.mha1(self.inducing.repeat(x.size(0),1,1), x, x)
        H = self.ln1(H)
        out, _ = self.mha2(x, H, H)
        out = self.ln2(out)
        return x + out


class PMA(nn.Module):
    def __init__(self, dim, heads):
        super().__init__()
        self.seed = nn.Parameter(torch.randn(1, 1, dim))
        self.mha = nn.MultiheadAttention(dim, heads, batch_first=True)

    def forward(self, x):
        out, _ = self.mha(self.seed.repeat(x.size(0),1,1), x, x)
        return out.mean(dim=1)


class TrueSetTransformer(BaseEstimator, RegressorMixin):
    def __init__(self, input_dim=768, hidden_dim=128, heads=4, num_inducing=32,
                 epochs=50, batch_size=128, lr=1.5e-3, patience=10):
        
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.num_inducing = num_inducing
        self.epochs = epochs
        self.batch_size = batch_size
        self.lr = lr
        self.patience = patience
        self.device = device

    def fit(self, X, y, eval_set=None):
        self.model = nn.Sequential(
            nn.Linear(self.input_dim, self.hidden_dim),
            ISAB(self.hidden_dim, self.heads, num_inducing=self.num_inducing),
            ISAB(self.hidden_dim, self.heads, num_inducing=self.num_inducing),
            PMA(self.hidden_dim, self.heads),
            nn.Linear(self.hidden_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        ).to(self.device)

        opt = torch.optim.AdamW(self.model.parameters(), lr=self.lr)
        loss_fn = nn.MSELoss()

        # 1. FAST GPU LOADING
        X_t = torch.as_tensor(X, dtype=torch.float32, device=self.device)
        y_t = torch.as_tensor(y, dtype=torch.float32, device=self.device).view(-1, 1)

        if eval_set:
            X_val_t = torch.as_tensor(eval_set[0], dtype=torch.float32, device=self.device)
            y_val_t = torch.as_tensor(eval_set[1], dtype=torch.float32, device=self.device).view(-1, 1)

        best_loss = float("inf")
        best_state = copy.deepcopy(self.model.state_dict())
        patience_counter = 0
        n_samples = X_t.size(0)
        self.history = {"train_loss": [], "val_loss": []}

        for epoch in range(self.epochs):
            self.model.train()
            
            perm = torch.randperm(n_samples, device=self.device)
            epoch_train_loss = 0.0
            
            for i in range(0, n_samples, self.batch_size):
                idx = perm[i:i+self.batch_size]
                xb, yb = X_t[idx], y_t[idx]
                
                opt.zero_grad()
                pred = self.model(xb)
                loss = loss_fn(pred, yb)
                loss.backward()
                opt.step()
                
                epoch_train_loss += loss.item() * xb.size(0)
                
            epoch_train_loss /= n_samples # THE FIX: Smooth, full-epoch loss

            # 3. EVALUATION
            if eval_set:
                self.model.eval()
                val_loss = 0.0
                with torch.no_grad():
                    # Chunked eval to prevent VRAM spikes
                    for i in range(0, X_val_t.size(0), self.batch_size):
                        xb_v = X_val_t[i:i+self.batch_size]
                        yb_v = y_val_t[i:i+self.batch_size]
                        val_loss += loss_fn(self.model(xb_v), yb_v).item() * xb_v.size(0)
                current_loss = val_loss / X_val_t.size(0)
            else:
                current_loss = epoch_train_loss # THE FIX: Accurate final model stopping metric

            self.history["train_loss"].append(epoch_train_loss)
            if eval_set:
                self.history["val_loss"].append(current_loss)
                
            # EARLY STOPPING LOGIC
            if current_loss < best_loss:
                best_loss = current_loss
                best_state = copy.deepcopy(self.model.state_dict())
                patience_counter = 0
            else:
                patience_counter += 1

            if patience_counter >= self.patience:
                break

        # Load best weights
        self.model.load_state_dict(best_state)
        
        # Flush Tensors
        del X_t, y_t
        if eval_set:
            del X_val_t, y_val_t
        torch.cuda.empty_cache()

        return self

    def predict(self, X):
        self.model.eval()
        X_t = torch.as_tensor(X, dtype=torch.float32, device=self.device)
        preds = []
        
        with torch.no_grad():
            for i in range(0, X_t.size(0), self.batch_size):
                xb = X_t[i:i+self.batch_size]
                preds.append(self.model(xb).cpu().numpy())
                
        # OOM PREVENTION
        del X_t
        torch.cuda.empty_cache()
        
        return np.vstack(preds).flatten()


X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_set, y_train,
    test_size=0.2,
    random_state=seed
)

model = TrueSetTransformer(input_dim=X_train_set.shape[2])
model.fit(X_tr, y_tr, eval_set=(X_val, y_val))

train_pred = model.predict(X_tr)
val_pred = model.predict(X_val)

train_r2 = r2_score(y_tr, train_pred)
val_r2 = r2_score(y_val, val_pred)

train_rmse = np.sqrt(mean_squared_error(y_tr, train_pred))
val_rmse = np.sqrt(mean_squared_error(y_val, val_pred))

gap = train_r2 - val_r2

# OUTPUT SUMMARY
print("\n" + "="*80)
print(" SET TRANSFORMER - BASELINE PERFORMANCE")
print("="*80)
print(f"Train R²   : {train_r2:.4f}")
print(f"Val   R²   : {val_r2:.4f}")
print(f"Gap        : {gap:.4f}")
print(f"Train RMSE : {train_rmse:.4f}")
print(f"Val   RMSE : {val_rmse:.4f}")
print("="*80)

# STORE FOR FINAL TABLE
results.append({
    "Model": "Set Transformer",
    "Train_R2": train_r2,
    "Val_R2": val_r2,
    "Gap_R2": gap,
    "Train_RMSE": train_rmse,
    "Val_RMSE": val_rmse
})


 SET TRANSFORMER - BASELINE PERFORMANCE
Train R²   : 0.5073
Val   R²   : 0.4498
Gap        : 0.0576
Train RMSE : 0.2344
Val   RMSE : 0.2534


In [7]:
import pandas as pd

results_df = pd.DataFrame(results)

print("\n" + "="*90)
print(" FINAL MODEL COMPARISON SUMMARY (BASELINE MODELS)")
print("="*90)

print(results_df.to_markdown(index=False, floatfmt=".4f"))


 FINAL MODEL COMPARISON SUMMARY (BASELINE MODELS)
| Model             |   Train_R2 |   Val_R2 |   Gap_R2 |   Train_RMSE |   Val_RMSE |
|:------------------|-----------:|---------:|---------:|-------------:|-----------:|
| Linear Regression |     0.3461 |   0.3198 |   0.0263 |       0.2713 |     0.2766 |
| LightGBM          |     0.5644 |   0.4814 |   0.0830 |       0.2214 |     0.2415 |
| CatBoost          |     0.6229 |   0.4922 |   0.1308 |       0.2060 |     0.2390 |
| SVR               |     0.7540 |   0.4746 |   0.2794 |       0.1664 |     0.2431 |
| Set Transformer   |     0.5073 |   0.4498 |   0.0576 |       0.2344 |     0.2534 |
